# Wannier-based embedding model for exchange interactions

This notebook implements the Wannier-based *ab initio* embedding model using
the bundled Wannier90 and VASP files. It builds a clean-host Green function,
extracts a localized spin-dependent Fe scattering potential from matched
4x4 supercells, and evaluates the two-impurity exchange interaction.

### Physical and numerical conventions

- Energies are in $\mathrm{eV}$ during the calculation and in $\mathrm{meV}$ in the final $J$ values.
- Lengths are in Angstrom ($\text{\AA}$).
- With fractional reciprocal coordinates, the Fourier convention is

  $$
  H(\mathbf{k})=\sum_{\mathbf{R}}
  e^{+2\pi i\,\mathbf{k}\cdot\mathbf{R}}H(\mathbf{R}).
  $$
- The calculation is collinear and has no spin-orbit coupling. Files ending in
  `.1_*` and `.2_*` are the majority- and minority-spin channels, respectively.
- The Heisenberg convention and the corresponding exchange definition are

  $$
  \mathcal{H}=-J\,\mathbf{e}_1\cdot\mathbf{e}_2,\qquad
  J=\frac{\Omega_{\mathrm{AFM}}-\Omega_{\mathrm{FM}}}{2}.
  $$

  Positive $J$ therefore favors parallel moments.
- The zero-temperature Lloyd/Krein convention is

  $$
  \Omega_X=\frac{1}{\pi}\int^{\mu}
  \operatorname{Im}\!\left[\ln\det(\cdots)\right]\,\mathrm{d}E.
  $$

### Equation notation

| Symbol | Meaning |
|---|---|
| $\mathbf{k}$ | Fractional reciprocal-space wave vector; its components are measured in the reciprocal lattice basis |
| $\mathbf{R}$ | Integer real-space cell translation; in this notebook it refers to the lattice of the Hamiltonian being used |
| $H(\mathbf{R})$, $H(\mathbf{k})$ | Real-space and Bloch representations of the same one-electron Hamiltonian |
| $J$, $\mathbf e_1$, $\mathbf e_2$ | Exchange constant and unit vectors along the two Fe moments |
| $\Omega_X$, $X$, $\mu$ | Electronic grand-potential change, magnetic configuration ($X=\mathrm{FM}$ or $\mathrm{AFM}$), and host chemical potential |

### Bundled inputs

| Role | Files relative to the project root |
|---|---|
| Clean nonmagnetic 1x1 host, 11 Wannier orbitals | `data/host_wannier/wannier90_hr.dat.gz`, `wannier90_centres.xyz`, `POSCAR` |
| Fe@4x4 supercell, 181 orbitals per spin | `data/fe4x4_wannier/wannier90.{1,2}_hr.dat.gz`, `wannier90.{1,2}_centres.xyz`, `POSCAR` |
| Matched Fe-free 4x4 twin, 176 orbitals per spin | `data/pristine4_wannier/wannier90.{1,2}_hr.dat.gz`, `wannier90.{1,2}_centres.xyz`, `POSCAR` |

Original OSCAR locations recorded with this dataset:

- `fullrelax_host_20260716/z0/host1x1/wannier`
- `nmgeom_j_20260712/z0/{fe4x4,pristine4_defgeom}/wannier`

### Variable-name convention

| Name | Meaning |
|---|---|
| `ROOT` | Project directory that contains `data/` |
| `HOSTW` | Directory containing the clean 1x1 host Wannier files |
| `FEW` | Directory containing the Fe@4x4 spin-resolved Wannier files |
| `PRW` | Directory containing the matched Fe-free 4x4 twin files |


In [1]:
import os
from pathlib import Path

import numpy as np


def find_project_root():
    """Locate the project directory that contains the bundled input data.

    Inputs:
        None. The search uses the current directory, its parents, and the
        optional Jupyter session path in ``JPY_SESSION_NAME``.

    Returns:
        pathlib.Path: Directory containing ``data/host_wannier``.

    Raises:
        RuntimeError: If no suitable project directory is found.
    """
    candidates = [Path.cwd(), *Path.cwd().parents]
    session_name = os.environ.get("JPY_SESSION_NAME", "")
    if session_name:
        session_path = Path(session_name).resolve()
        candidates.extend([session_path.parent, *session_path.parents])

    for candidate in candidates:
        if (candidate / "data" / "host_wannier").is_dir():
            return candidate

    raise RuntimeError(
        "Cannot find data/: start the notebook from the project directory "
        "or change the kernel working directory to the folder containing data/."
    )


ROOT = find_project_root()
DATA = ROOT / "data"
HOSTW = DATA / "host_wannier"          # Clean 1x1 host Wannier files.
FEW = DATA / "fe4x4_wannier"           # Spin-resolved Fe@4x4 Wannier files.
PRW = DATA / "pristine4_wannier"       # Matched Fe-free 4x4 twin files.
print("project root =", ROOT)

# Global numerical controls.
ETA = 0.06          # eV; positive imaginary part of the retarded Green function.
RCUT = 5.3          # Angstrom; radius of the Fe-centered scattering cluster.
NE = 180            # Energy points on the interval [mu - 7.5 eV, mu].
NK = 180            # Dense host Green-function mesh: NK x NK k points.
LT = 18             # Real-space bookkeeping grid: LT x LT primitive cells.
FILLING = 6.5       # Host electrons per spin and primitive cell; fixes mu.
OFFSETS = (2, 3, 4, 5, 6)  # Fe-Fe separations in units of primitive vector a1.

# Fail early if any raw input required by the notebook is missing.
required_files = [
    HOSTW / "wannier90_hr.dat.gz",
    HOSTW / "wannier90_centres.xyz",
    HOSTW / "POSCAR",
    FEW / "wannier90.1_hr.dat.gz",
    FEW / "wannier90.2_hr.dat.gz",
    FEW / "wannier90.1_centres.xyz",
    FEW / "wannier90.2_centres.xyz",
    FEW / "POSCAR",
    PRW / "wannier90.1_hr.dat.gz",
    PRW / "wannier90.2_hr.dat.gz",
    PRW / "wannier90.1_centres.xyz",
    PRW / "wannier90.2_centres.xyz",
    PRW / "POSCAR",
]
for path in required_files:
    assert path.is_file(), f"missing raw input: {path}"

# NumPy >=2.0 renamed np.trapz to np.trapezoid; keep one local API.
trapezoid = getattr(np, "trapezoid", None) or np.trapz
print("all bundled input files present")


project root = C:\Users\39381\Downloads\notebook_j_pipeline_20260721\prts-wannier-exchange
all bundled input files present


## 1. Wannier90 and structure-file readers

### `*_hr.dat` convention

After the comment line, a Wannier90 Hamiltonian file contains the number of
Wannier functions `nw`, the number of stored lattice vectors `nr`, `nr`
Wigner-Seitz degeneracy weights, and $n_r n_w^2$ matrix-element rows. Each data
row contains seven fields:

| Field | Meaning |
|---|---|
| `Rx Ry Rz` | Integer components of the lattice translation $\mathbf R=R_x\mathbf A_1+R_y\mathbf A_2+R_z\mathbf A_3$ |
| `j i` | One-based Wannier-orbital labels in the file's native loop order; the reader transposes this order so NumPy entry `[i,j]` follows the convention below |
| `Re Im` | Real and imaginary parts of the complex Hamiltonian matrix element |

Wannier90's file loop order would be transposed if read directly as `[i,j]`.
`read_hr` therefore transposes each block and uses the matrix-element convention

$$
H_{ij}(\mathbf{R})=\langle w_{i,\mathbf{0}}|\hat H|w_{j,\mathbf{R}}\rangle.
$$

It also divides by the Wigner-Seitz degeneracy, so the returned dictionary can
be Fourier transformed directly as

$$
H(\mathbf{k})=\sum_{\mathbf{R}}
e^{+2\pi i\,\mathbf{k}\cdot\mathbf{R}}H(\mathbf{R}).
$$

### Equation notation

| Symbol | Meaning |
|---|---|
| $n_w$, $n_r$ | Number of Wannier orbitals and number of stored lattice translations |
| $i$, $j$ | Row and column orbital indices after conversion to the notebook convention |
| $|w_{i,\mathbf0}\rangle$, $|w_{j,\mathbf R}\rangle$ | Wannier orbital $i$ in the reference cell and orbital $j$ in cell $\mathbf R$ |
| $\hat H$, $H_{ij}(\mathbf R)$ | Hamiltonian operator and its real-space matrix element |
| $\mathbf k$, $\mathbf R$ | Fractional reciprocal wave vector and integer real-space lattice translation |

`*_centres.xyz` supplies Cartesian Wannier centers, while `POSCAR` supplies the
lattice, atom labels, and fractional atomic coordinates.

### Variable-name convention

| Name | Meaning |
|---|---|
| `nw` | Number of Wannier functions; also the dimension of each `H(R)` block |
| `nr` | Number of stored real-space lattice vectors `R` |
| `deg` | Wigner-Seitz degeneracy weight associated with each stored `R` |
| `lat` | `3x3` lattice matrix whose rows are Cartesian lattice vectors |
| `frac` | `natoms x 3` fractional atomic coordinates from POSCAR |


In [2]:
# Cell 2: file readers
import gzip


def read_hr(path):
    """Read a plain or gzip-compressed Wannier90 ``*_hr.dat`` file.

    Inputs:
        path (str or pathlib.Path): Hamiltonian file; a ``.gz`` suffix selects
            transparent gzip decompression.

    Returns:
        dict: ``{(Rx,Ry,Rz): H(R)}``, where each value is a complex
        ``(nw,nw)`` array satisfying
        ``H[R][i,j] = <w_i,0|H|w_j,R>``. Wigner-Seitz degeneracy weights are
        already divided out.
    """
    opener = (lambda q: gzip.open(q, "rt")) if str(path).endswith(".gz") else open
    with opener(path) as handle:
        handle.readline()  # Free-form comment line.
        nw = int(handle.readline().split()[0])
        nr = int(handle.readline().split()[0])

        # Degeneracy weights can span multiple lines in groups of at most 15.
        degeneracies = []
        while len(degeneracies) < nr:
            degeneracies.extend(int(x) for x in handle.readline().split())
        degeneracies = np.asarray(degeneracies[:nr], dtype=float)
        data = np.loadtxt(handle)

    data = data.reshape(nr, nw, nw, 7)
    hamiltonian = {}
    for ir in range(nr):
        R = tuple(int(data[ir, 0, 0, component]) for component in (0, 1, 2))
        # Wannier90 loops fastest over its first orbital index; transpose to
        # the notebook convention H[R][i,j] = <w_i,0|H|w_j,R>.
        block = data[ir, :, :, 5] + 1j * data[ir, :, :, 6]
        hamiltonian[R] = block.T / degeneracies[ir]
    return hamiltonian


def read_poscar(path):
    """Read the restricted direct-coordinate POSCAR format used here.

    Inputs:
        path (str or pathlib.Path): POSCAR file with Direct coordinates and no
            Selective-dynamics line.

    Returns:
        tuple: ``(lattice, labels, frac)`` where ``lattice`` is ``(3,3)`` in
        Angstrom with lattice vectors as rows, ``labels`` contains one species
        string per atom, and ``frac`` is a ``(natoms,3)`` coordinate array.
    """
    lines = Path(path).read_text().splitlines()
    scale = float(lines[1].split()[0])
    lattice = np.array(
        [[float(x) for x in lines[i].split()[:3]] for i in (2, 3, 4)]
    ) * scale
    species = lines[5].split()
    counts = [int(x) for x in lines[6].split()]
    labels = [symbol for symbol, count in zip(species, counts) for _ in range(count)]
    assert lines[7].strip().lower().startswith("d"), "expected Direct coordinates"

    natoms = sum(counts)
    frac = np.array(
        [[float(x) for x in lines[8 + i].split()[:3]] for i in range(natoms)]
    )
    return lattice, labels, frac


def read_centres(path, count):
    """Read Cartesian Wannier centers from a ``*_centres.xyz`` file.

    Inputs:
        path (str or pathlib.Path): Wannier90 centers file.
        count (int): Number of rows beginning with ``X`` to read.

    Returns:
        numpy.ndarray: Cartesian center coordinates with shape ``(count,3)``
        in Angstrom.
    """
    rows = []
    for line in Path(path).read_text().splitlines()[2:]:
        fields = line.split()
        if fields and fields[0] == "X":
            rows.append([float(value) for value in fields[1:4]])
            if len(rows) == count:
                break

    assert len(rows) == count, f"{path}: found {len(rows)} centres, expected {count}"
    return np.asarray(rows)


print("readers defined")


readers defined


## 2. Geometry and orbital layout

The in-plane $4\times4$ supercell vectors $\mathbf{A}_1$ and $\mathbf{A}_2$
are read from the Fe-supercell POSCAR. The primitive vectors satisfy
$\mathbf{a}_1=\mathbf{A}_1/4$ and $\mathbf{a}_2=\mathbf{A}_2/4$. The out-of-plane
lattice vector is used only through the Cartesian atomic heights needed for
distance-based selections.

The 181-orbital Fe-supercell basis follows the Wannier projection order:

- 16 Ta atoms x 5 d orbitals: indices `0:80`;
- one Fe atom x 5 d orbitals: indices `80:85`, at primitive-cell coordinate `(2,2)`;
- 16 bottom-layer S atoms x 3 p orbitals: indices `85:133`;
- 16 top-layer S atoms x 3 p orbitals: indices `133:181`.

The 176-orbital pristine twin has the same ordering with the Fe block removed.
The primitive 11-orbital host order is Ta d `0:5`, bottom-S p `5:8`, and top-S
p `8:11`. `Atom.start` and `Atom.stop` always form a half-open orbital range.

Both S sublattices have the same in-plane basis offset $(1/3,2/3)$. The fixed
fractional heights are used only for real-space distances, not to rebuild the
Hamiltonian.

### Geometry notation

| Symbol | Meaning |
|---|---|
| $\mathbf A_1$, $\mathbf A_2$ | Cartesian in-plane lattice vectors of the $4\times4$ source supercell |
| $\mathbf a_1$, $\mathbf a_2$ | Primitive in-plane lattice vectors, one quarter of the corresponding source-supercell vectors |
| $(c_a,c_b)$ | Integer primitive-cell coordinates stored as `Atom.ca` and `Atom.cb`; these are cell labels, not Cartesian coordinates |
| `[start:stop]` | Half-open orbital-index interval: `start` is included and `stop` is excluded |
| d, p | Ta/Fe d-like and S p-like Wannier orbitals; they label orbital character rather than scalar variables |

### Variable-name convention

| Name | Meaning |
|---|---|
| `lat4` | `3x3` lattice matrix of the Fe@4x4 supercell |
| `frac4` | Fractional coordinates of atoms in the Fe@4x4 POSCAR |
| `A1`, `A2` | In-plane 4x4 supercell vectors in Angstrom |
| `a1`, `a2` | In-plane primitive-cell vectors, equal to `A1/4` and `A2/4` |
| `ATOM4` | Atom-to-orbital ledger for the 181-orbital Fe@4x4 basis |


In [3]:
# Cell 3: geometry constants and orbital ledgers
from dataclasses import dataclass


lat4, labels4, frac4 = read_poscar(FEW / "POSCAR")
A1, A2 = lat4[0, :2].copy(), lat4[1, :2].copy()  # In-plane 4x4 vectors.
a1, a2 = A1 / 4.0, A2 / 4.0                    # Primitive in-plane vectors.
C_AX = float(np.linalg.norm(lat4[2]))            # Out-of-plane lattice length.

# In-plane primitive-cell offsets and fixed fractional heights for each sublattice.
BASIS_2D = {
    "Ta": np.array([0.0, 0.0]),
    "S_bot": np.array([1 / 3, 2 / 3]),
    "S_top": np.array([1 / 3, 2 / 3]),
    "Fe": np.array([0.0, 0.0]),
}
Z_FRAC = {"Ta": 0.09630, "S_bot": 0.04833, "S_top": 0.14463, "Fe": 0.17619}
TYPE_WF_1 = {"Ta": (0, 5), "S_bot": (5, 8), "S_top": (8, 11)}


@dataclass(frozen=True)
class Atom:
    """Describe one atom and its half-open orbital block.

    Attributes:
        typ: ``Ta``, ``S_bot``, ``S_top``, or ``Fe``.
        ca, cb: Integer primitive-cell coordinates. Their reference grid is
            the 4x4 source cell for ``ATOM4`` and the LT x LT bookkeeping grid
            for the later host atom list.
        start, stop: Half-open orbital range ``[start, stop)`` in the relevant
            basis.
    """

    typ: str
    ca: int
    cb: int
    start: int
    stop: int


# Fixed 181-orbital ledger of the Fe@4x4 Wannier basis.
ATOM4 = []
for i in range(16):
    ATOM4.append(Atom("Ta", i // 4, i % 4, 5 * i, 5 * (i + 1)))
ATOM4.append(Atom("Fe", 2, 2, 80, 85))
for j in range(16):
    ATOM4.append(Atom("S_bot", j // 4, j % 4, 85 + 3 * j, 85 + 3 * (j + 1)))
for j in range(16):
    ATOM4.append(Atom("S_top", j // 4, j % 4, 133 + 3 * j, 133 + 3 * (j + 1)))


def position_3d(typ, cell):
    """Return the Cartesian position of a sublattice site.

    Inputs:
        typ (str): Sublattice label used by ``BASIS_2D`` and ``Z_FRAC``.
        cell (array-like): Two integer primitive-cell coordinates ``(ca,cb)``.

    Returns:
        numpy.ndarray: Cartesian ``(x,y,z)`` position in Angstrom.

    Notes:
        This helper is used only for real-space distance selections. It does
        not rebuild or modify any Wannier Hamiltonian matrix element.
    """
    primitive_position = np.asarray(cell, dtype=float) + BASIS_2D[typ]
    xy = primitive_position[0] * a1 + primitive_position[1] * a2
    return np.array([xy[0], xy[1], Z_FRAC[typ] * C_AX])


print(f"A1 = {A1}, |a1| = {np.linalg.norm(a1):.6f} A, c = {C_AX:.6f} A")


A1 = [13.35408115  0.        ], |a1| = 3.338520 A, c = 32.551073 A


## 3. Restore Wannier functions to their atomic home cells

Wannier90 may represent a Wannier function by a periodic image of its center
while the orbital ledger still associates that function with the atom in the
nominal cell. This is a basis-label mismatch, not a change in the physical
band structure.

For orbital $i$ attached to atom $a$, `infer_wraps` first expresses the
center-to-atom displacement in lattice coordinates and then rounds its two
in-plane components:

$$
\mathbf{f}_i=\mathbf{L}^{-T}
\left(\mathbf{r}^{\mathrm{WC}}_i-\mathbf{r}_a\right),\qquad
\mathbf{w}_i=\operatorname{nint}\!\left[(f_{i,x},f_{i,y})\right].
$$

Only the two in-plane components are wrapped. Orbitals belonging to the same
atom are required to have the same integer wrap. The reported residual checks
that the inferred fractional displacement is far from the ambiguous half-cell
boundary.

The exact relabeling is

$$
H'_{ij}\!\left(\mathbf{R}+\mathbf{w}_j-\mathbf{w}_i\right)
=H_{ij}(\mathbf{R}).
$$

In reciprocal space this is a diagonal unitary gauge transformation, so the
eigenvalues of $H(\mathbf{k})$ are unchanged. Re-homing matters for real-space
slicing, basis mapping, and truncation because those operations depend explicitly
on $\mathbf{R}$.

### Equation notation

| Symbol | Meaning |
|---|---|
| $i$, $j$, $a$ | Orbital indices $i,j$ and the atom label $a$ to which an orbital block is assigned |
| $\mathbf L$ | $3\times3$ lattice matrix with Cartesian lattice vectors as rows |
| $\mathbf r_i^{\mathrm{WC}}$, $\mathbf r_a$ | Cartesian position of Wannier center $i$ and Cartesian position of atom $a$ |
| $\mathbf f_i$, $\mathbf w_i$ | Fractional center-to-atom displacement and its nearest integer in-plane cell offset |
| $\operatorname{nint}$, $\mathbf R$ | Componentwise nearest-integer operation and an integer lattice translation |

### Variable-name convention

| Name | Meaning |
|---|---|
| `centres` | `nw x 3` Cartesian Wannier-center coordinates |
| `atom_cart` | `natoms x 3` Cartesian atomic coordinates |
| `blocks` | Triples `(atom_index, orbital_start, orbital_stop)` |
| `wraps` | `nw x 2` integer in-plane cell offsets, one row per orbital |
| `H` | Dictionary mapping integer lattice vectors `R` to complex matrices `H(R)` |


In [4]:
# Cell 4: infer home-cell labels and re-home real-space Hamiltonians
from collections import defaultdict


def infer_wraps(centres, atom_cart, blocks, lattice):
    """Infer each orbital's integer in-plane home-cell offset.

    Inputs:
        centres (numpy.ndarray): Cartesian Wannier centers, shape ``(nw,3)``.
        atom_cart (numpy.ndarray): Cartesian atomic positions, shape
            ``(natoms,3)``.
        blocks (list): ``(atom_index,start,stop)`` triples assigning complete
            orbital blocks to atoms.
        lattice (numpy.ndarray): ``(3,3)`` lattice matrix with vectors as rows.

    Returns:
        numpy.ndarray: Integer offsets ``wraps`` with shape ``(nw,2)``.

    Raises:
        AssertionError: If orbitals assigned to one atom infer different wraps.
    """
    nw = max(stop for _, _, stop in blocks)
    wraps = np.zeros((nw, 2), dtype=int)
    worst = 0.0

    for atom_index, start, stop in blocks:
        atom_wraps = []
        for orbital in range(start, stop):
            # Solve lattice.T @ fractional = center - atom_position.
            fractional = np.linalg.solve(
                lattice.T,
                centres[orbital] - atom_cart[atom_index],
            )
            wrap = np.rint(fractional[:2]).astype(int)
            residual = np.max(np.abs(fractional[:2] - wrap))
            worst = max(worst, float(residual))
            wraps[orbital] = wrap
            atom_wraps.append(tuple(wrap))

        assert len(set(atom_wraps)) == 1, (
            f"wrap mismatch inside atom {atom_index}: {atom_wraps}"
        )

    print(f"  worst fractional residual = {worst:.4f} (must be << 0.5)")
    return wraps


def rehome(H, wraps):
    """Apply the exact real-space gauge relabeling to an R-space Hamiltonian.

    Inputs:
        H (dict): ``{R: matrix}`` with complex ``(nw,nw)`` blocks.
        wraps (numpy.ndarray): Integer in-plane offsets with shape ``(nw,2)``.

    Returns:
        dict: Re-homed blocks satisfying
        ``H'(R + w_j - w_i)[i,j] = H(R)[i,j]``. Numerically zero blocks with
        maximum magnitude at most ``1e-14`` are omitted.
    """
    groups = defaultdict(list)
    for orbital, wrap in enumerate(wraps):
        groups[tuple(int(value) for value in wrap)].append(orbital)

    nw = len(wraps)
    output = defaultdict(lambda: np.zeros((nw, nw), dtype=complex))
    for (Rx, Ry, Rz), matrix in H.items():
        for wi, rows in groups.items():
            for wj, cols in groups.items():
                new_R = (Rx + wj[0] - wi[0], Ry + wj[1] - wi[1], Rz)
                block_index = np.ix_(rows, cols)
                output[new_R][block_index] += matrix[block_index]

    return {
        R: matrix
        for R, matrix in output.items()
        if np.max(np.abs(matrix)) > 1e-14
    }


def herm_residual(H):
    """Measure the maximum real-space Hermiticity violation.

    Inputs:
        H (dict): Real-space matrix dictionary ``{R: H(R)}``.

    Returns:
        float: Maximum element of ``|H(R) - H(-R)^dagger|`` over stored `R`.
    """
    zero = np.zeros_like(next(iter(H.values())))
    residuals = []
    for R, matrix in H.items():
        minus_R = (-R[0], -R[1], -R[2])
        partner = H.get(minus_R, zero).conj().T
        residuals.append(float(np.max(np.abs(matrix - partner))))
    return max(residuals)


print("re-homing machinery defined")


re-homing machinery defined


### 3a. Re-home and validate the clean 1x1 host

The host POSCAR contains one Ta atom and two S atoms. The lower and upper S
atoms are identified by Cartesian height, then associated with the fixed
11-orbital host order. All six S-centered Wannier functions are known to use a
neighboring-cell image in the raw data; `rehome` restores their intended
real-space labels.
The reported Hermiticity residual is an internal consistency check on the
re-homed real-space Hamiltonian; it does not compare against an external result.

### Variable-name convention

| Name | Meaning |
|---|---|
| `latH` | Primitive-host lattice matrix |
| `cartH` | Cartesian positions of the three host atoms, `fracH @ latH` |
| `blocksH` | Host atom-to-orbital mapping for Ta, bottom S, and top S |
| `wrapsH` | Integer home-cell offsets inferred for the 11 host orbitals |
| `H_host` | Re-homed clean-host real-space Hamiltonian dictionary |


In [5]:
# Cell 5: clean-host re-homing and Hermiticity check
latH, labH, fracH = read_poscar(HOSTW / "POSCAR")
cartH = fracH @ latH

# Identify the two S layers by Cartesian height. The orbital ranges follow the
# fixed primitive-host order Ta d, bottom-S p, top-S p.
iTa = labH.index("Ta")
iS = [index for index, label in enumerate(labH) if label == "S"]
iS_bot, iS_top = sorted(iS, key=lambda index: cartH[index, 2])
blocksH = [(iTa, 0, 5), (iS_bot, 5, 8), (iS_top, 8, 11)]

H_host_raw = read_hr(HOSTW / "wannier90_hr.dat.gz")
cenH = read_centres(HOSTW / "wannier90_centres.xyz", 11)
print("host wrap inference:")
wrapsH = infer_wraps(cenH, cartH, blocksH, latH)
print("  wraps per orbital:", wrapsH.tolist())
H_host = rehome(H_host_raw, wrapsH)
print(f"  host Hermiticity residual after re-homing: {herm_residual(H_host):.2e} eV")


host wrap inference:
  worst fractional residual = 0.0464 (must be << 0.5)
  wraps per orbital: [[0, 0], [0, 0], [0, 0], [0, 0], [0, 0], [0, -1], [0, -1], [0, -1], [0, -1], [0, -1], [0, -1]]
  host Hermiticity residual after re-homing: 0.00e+00 eV


## 4. Construct the spin-dependent Fe scattering potential

For each spin channel, the raw supercell perturbation is
$$
V_{\sigma}(\mathbf{R})=H_{\mathrm{Fe},\sigma}(\mathbf{R})
-\mathcal P\!\left[H_{\mathrm{pristine},\sigma}(\mathbf{R})\right].
$$

Here $\mathcal P$ is only a zero-padding and index-placement operation. The
pristine 176-orbital Hamiltonian is padded into the 181-orbital Fe layout:
Ta `0:80` maps to `0:80`, S `80:176` maps to `85:181`, and the Fe rows and
columns remain zero. Consequently, the Fe-host couplings and Fe d block come
entirely from the Fe calculation.

This basis padding is deliberately not called *embedding*. In this notebook,
impurity embedding refers to placing the extracted effective potential into an
arbitrary host environment or impurity separation later in the workflow.

Before subtraction, the code audits the common basis: both twins must have the
same lattice, host geometry, atom ordering, home-cell wraps, and closely
matching re-homed Wannier centers. Matching centers cannot prove equality of
orbital phases or local unitary gauges, so a far-field Hamiltonian audit is
performed in the next section.

Because the common host wraps are identical, linearity allows the code to
subtract the raw Hamiltonians first and re-home their difference once. If the
wraps differed, the two Hamiltonians would need separate re-homing before any
subtraction.

### Equation notation

| Symbol | Meaning |
|---|---|
| $\sigma$ | Collinear spin channel, represented by code key `1` or `2` |
| $\mathbf R$ | Translation of one complete $4\times4$ source supercell |
| $H_{\mathrm{Fe},\sigma}$ | 181-orbital Hamiltonian of the Fe-containing twin |
| $H_{\mathrm{pristine},\sigma}$ | 176-orbital Hamiltonian of the matched Fe-free twin |
| $\mathcal P$, $V_\sigma$ | Basis-padding operator and the resulting Fe-induced Hamiltonian difference |

### Variable-name convention

| Name | Meaning |
|---|---|
| `PRISTINE_SLOTS` | 176 host-orbital positions inside the 181-orbital Fe basis |
| `blocks4` | Atom-to-orbital ledger for the 181-orbital Fe supercell |
| `blocksP` | Atom-to-orbital ledger for the 176-orbital pristine twin |
| `host_orbs` | The 176 non-Fe orbital indices in the 181-orbital layout |
| `V` | Dictionary `V[spin][R]` containing the re-homed 181x181 perturbation blocks |


In [6]:
# Cell 6: assemble the raw Fe perturbation and audit the common twin basis
# Host-orbital destinations used only for zero-padding the pristine matrix;
# this index placement is not the later impurity-embedding operation.
PRISTINE_SLOTS = list(range(0, 80)) + list(range(85, 181))


def pad_pristine_to_fe_basis(Hp):
    """Pad a pristine 176-orbital Hamiltonian into the 181-orbital Fe basis.

    Inputs:
        Hp (dict): ``{R: matrix}`` with complex ``(176,176)`` blocks.

    Returns:
        dict: ``{R: matrix}`` with complex ``(181,181)`` blocks. Host-host
        entries preserve their pristine ordering; all Fe rows and columns are
        zero.
    """
    output = {}
    for R, matrix in Hp.items():
        padded = np.zeros((181, 181), dtype=complex)
        padded[np.ix_(PRISTINE_SLOTS, PRISTINE_SLOTS)] = matrix
        output[R] = padded
    return output


cart4 = frac4 @ lat4


def atom_index_4x4(atom):
    """Map an ``ATOM4`` entry to the corresponding Fe-POSCAR atom index.

    Inputs:
        atom (Atom): Source-ledger atom with 4x4 primitive-cell coordinates.

    Returns:
        int: POSCAR index in the order 16 Ta, 16 bottom S, 16 top S, one Fe.
    """
    ordinal = 4 * atom.ca + atom.cb
    if atom.typ == "Ta":
        return ordinal
    if atom.typ == "S_bot":
        return 16 + ordinal
    if atom.typ == "S_top":
        return 32 + ordinal
    if atom.typ == "Fe":
        return 48
    raise ValueError(f"unknown atom type: {atom.typ}")


blocks4 = [(atom_index_4x4(atom), atom.start, atom.stop) for atom in ATOM4]

# The pristine POSCAR contains the same 48 host atoms and no Fe atom.
latP, labP, fracP = read_poscar(PRW / "POSCAR")
cartP = fracP @ latP
blocksP = (
    [(i, 5 * i, 5 * (i + 1)) for i in range(16)]
    + [(16 + j, 80 + 3 * j, 80 + 3 * (j + 1)) for j in range(32)]
)

# Structural portion of the common-basis contract.
lat_twin_err = float(np.max(np.abs(lat4 - latP)))
geom_twin_err = float(np.max(np.linalg.norm(cart4[:48] - cartP, axis=1)))
labels_match = labels4[:48] == labP
print(
    f"BASIS AUDIT geometry: lattice max = {lat_twin_err:.3e} A, "
    f"host-atom max = {geom_twin_err:.3e} A, labels_match = {labels_match}"
)
assert labels_match, "Fe/pristine host atom ordering differs"
assert lat_twin_err < 1e-10 and geom_twin_err < 1e-8, (
    "Fe/pristine twins do not share the same host geometry"
)

host_orbs = np.asarray(PRISTINE_SLOTS, dtype=int)


def centres_rehomed(centres, wraps, lattice):
    """Remove integer in-plane cell offsets from Cartesian Wannier centers.

    Inputs:
        centres (numpy.ndarray): Cartesian centers, shape ``(nw,3)``.
        wraps (numpy.ndarray): Integer offsets, shape ``(nw,2)``.
        lattice (numpy.ndarray): Lattice vectors as rows, shape ``(3,3)``.

    Returns:
        numpy.ndarray: Re-homed Cartesian centers with shape ``(nw,3)``.
    """
    return (
        centres
        - wraps[:, 0, None] * lattice[0]
        - wraps[:, 1, None] * lattice[1]
    )


V = {}
for spin in (1, 2):
    Hfe = read_hr(FEW / f"wannier90.{spin}_hr.dat.gz")
    Hpr = read_hr(PRW / f"wannier90.{spin}_hr.dat.gz")
    cF = read_centres(FEW / f"wannier90.{spin}_centres.xyz", 181)
    cP = read_centres(PRW / f"wannier90.{spin}_centres.xyz", 176)

    print(f"spin {spin}: Fe wrap inference")
    wF = infer_wraps(cF, cart4, blocks4, lat4)
    print(f"spin {spin}: pristine wrap inference")
    wP = infer_wraps(cP, cartP, blocksP, latP)

    # Direct raw subtraction is valid only when common host orbitals use the
    # same integer cell-label convention in both twins.
    wP_in_fe_basis = np.zeros((181, 2), dtype=int)
    wP_in_fe_basis[PRISTINE_SLOTS] = wP
    mismatch_mask = np.any(
        wF[host_orbs] != wP_in_fe_basis[host_orbs],
        axis=1,
    )
    mismatch_count = int(np.sum(mismatch_mask))
    print(f"spin {spin}: twin wrap mismatches on host orbitals = {mismatch_count}")
    assert mismatch_count == 0, "twin wrap inconsistency"

    # Re-home centers only for this correspondence audit; the Hamiltonians are
    # still raw at this point and their difference is re-homed below.
    cF_home = centres_rehomed(cF, wF, lat4)[host_orbs]
    cP_home = centres_rehomed(cP, wP, latP)
    centre_delta = np.linalg.norm(cF_home - cP_home, axis=1)
    centre_max = float(np.max(centre_delta))
    centre_rms = float(np.sqrt(np.mean(centre_delta**2)))
    print(
        f"spin {spin}: common-centre delta max/rms = "
        f"{centre_max:.4f}/{centre_rms:.4f} A"
    )
    assert centre_max < 0.25, (
        "common Wannier centres are inconsistent; twin orbital order/gauge may differ"
    )

    # The common wraps make rehome(Hfe - Hpr) exactly equivalent to separately
    # re-homing both twins and subtracting the results.
    Hpr_padded = pad_pristine_to_fe_basis(Hpr)
    zero181 = np.zeros((181, 181), dtype=complex)
    all_R = set(Hfe) | set(Hpr_padded)
    Vraw = {
        R: Hfe.get(R, zero181) - Hpr_padded.get(R, zero181)
        for R in all_R
    }
    V[spin] = rehome(Vraw, wF)
    print(
        f"spin {spin}: V Hermiticity residual = "
        f"{herm_residual(V[spin]):.2e} eV"
    )


BASIS AUDIT geometry: lattice max = 0.000e+00 A, host-atom max = 1.850e-11 A, labels_match = True
spin 1: Fe wrap inference
  worst fractional residual = 0.0137 (must be << 0.5)
spin 1: pristine wrap inference
  worst fractional residual = 0.0125 (must be << 0.5)
spin 1: twin wrap mismatches on host orbitals = 0
spin 1: common-centre delta max/rms = 0.1427/0.0397 A
spin 1: V Hermiticity residual = 0.00e+00 eV
spin 2: Fe wrap inference
  worst fractional residual = 0.0132 (must be << 0.5)
spin 2: pristine wrap inference
  worst fractional residual = 0.0125 (must be << 0.5)
spin 2: twin wrap mismatches on host orbitals = 0
spin 2: common-centre delta max/rms = 0.1023/0.0297 A
spin 2: V Hermiticity residual = 0.00e+00 eV


### 4a. Far-field gauge audit and Fe/pristine energy alignment

The Fe and pristine twins are independent DFT calculations, so their arbitrary
one-electron energy zeros can differ. Nine Ta sites at least 5.5 Angstrom from
Fe provide a reference. For each spin, the mean of their five on-site d levels
defines

$$
c_{\sigma}=\frac{1}{N_{\mathrm{far}}}
\sum_{a\in\mathrm{far\ Ta}}\frac{1}{5}\sum_{m=1}^{5}
\left[V_{\sigma}(\mathbf{0})\right]_{am,am}.
$$

After temporarily removing $c_{\sigma}$ from the far-host $\mathbf{R}=\mathbf{0}$
diagonal, the expected far-field behavior is

$$
V_{\sigma}^{\mathrm{far}}(\mathbf{R})\simeq
\begin{cases}
c_{\sigma}I, & \mathbf{R}=\mathbf{0},\\
0, & \mathbf{R}\ne\mathbf{0}.
\end{cases}
$$

The residual is tested over every stored $\mathbf{R}$.
Large off-diagonal or intercell residuals would indicate an orbital phase,
permutation, or local-unitary mismatch rather than a localized impurity.

The common shift

$$
c_{\mathrm{common}}=\frac{c_1+c_2}{2}
$$

is permanently removed from all 181 on-site diagonals in both spin channels.
Using a common value preserves the spin-dependent part $c_1-c_2$.

This operation aligns Fe@4x4 to its pristine 4x4 twin. It does not independently
establish the absolute energy alignment between the pristine 4x4 reference and
the separate 1x1 host. In particular, the absolute Fe `Hdd` block later assumes
that those references are compatible; that assumption should be documented or
audited with a common vacuum/electrostatic reference when required.

### Equation notation

| Symbol | Meaning |
|---|---|
| $c_\sigma$, $c_1$, $c_2$ | Far-field energy-zero offset for spin $\sigma$ and its values in the two spin channels |
| $N_{\mathrm{far}}$ | Number of distant Ta sites used to estimate the energy-zero offset |
| $a$, $m$ | Distant-Ta site label and one of its five d-orbital labels |
| $I$ | Identity matrix on the selected far-host orbital subspace |
| $V_\sigma^{\mathrm{far}}(\mathbf R)$ | Far-host-to-far-host submatrix of the Fe-induced perturbation at translation $\mathbf R$ |

### Variable-name convention

| Name | Meaning |
|---|---|
| `far_ta` | Nine Ta atoms whose minimum-image distance from Fe is at least 5.5 Angstrom |
| `cs` | Dictionary of per-spin far-Ta offsets `c_sigma` |
| `far_host_orbs` | Orbital indices on all far Ta and S host atoms used by the gauge audit |
| `c_common` | Spin-averaged Fe/pristine energy-zero correction |
| `V[spin]` | Re-homed and, after this section, energy-aligned scattering potential |


In [7]:
# Cell 6 continued: far-field audit and Fe/pristine energy-zero alignment
def dmin_supercell(p, q):
    """Return the minimum in-plane-periodic distance between two points.

    Inputs:
        p, q (array-like): Cartesian three-vectors in Angstrom.

    Returns:
        float: Minimum Euclidean distance over the 3x3 set of neighboring
        4x4-supercell images in the two periodic in-plane directions.
    """
    A1_3d = np.append(A1, 0.0)
    A2_3d = np.append(A2, 0.0)
    best = np.inf
    for m in (-1, 0, 1):
        for n in (-1, 0, 1):
            image_displacement = p + m * A1_3d + n * A2_3d - q
            best = min(best, float(np.linalg.norm(image_displacement)))
    return best


# Use distant Ta on-site levels to estimate the arbitrary Fe/pristine offset.
fe_pos = position_3d("Fe", np.array([2, 2]))
far_ta = [
    atom
    for atom in ATOM4
    if atom.typ == "Ta"
    and dmin_supercell(
        position_3d("Ta", np.array([atom.ca, atom.cb])),
        fe_pos,
    ) >= 5.5
]
print(
    f"far-Ta sites (d >= 5.5 A): {len(far_ta)} -> "
    f"{[(atom.ca, atom.cb) for atom in far_ta]}"
)
assert len(far_ta) == 9

# c_sigma is the far-Ta mean shift for each spin channel.
cs = {}
for spin in (1, 2):
    onsite_diagonal = np.real(np.diag(V[spin][(0, 0, 0)]))
    site_means = [
        float(np.mean(onsite_diagonal[atom.start:atom.stop]))
        for atom in far_ta
    ]
    cs[spin] = float(np.mean(site_means))
    print(f"spin {spin}: c_sigma = {cs[spin]:+.6f} eV")

# A phase, permutation, or local-unitary mismatch would leave hopping-scale
# residuals far from Fe instead of a scalar on-site offset.
far_host_atoms = [
    atom
    for atom in ATOM4
    if atom.typ != "Fe"
    and dmin_supercell(
        position_3d(atom.typ, np.array([atom.ca, atom.cb])),
        fe_pos,
    ) >= 5.5
]
far_host_orbs = np.asarray(
    [
        orbital
        for atom in far_host_atoms
        for orbital in range(atom.start, atom.stop)
    ],
    dtype=int,
)
print(
    f"BASIS AUDIT far-host set: {len(far_host_atoms)} atoms, "
    f"{len(far_host_orbs)} orbitals"
)

# After removing the arbitrary scalar energy-zero offset, test whether every
# far-host-to-far-host element of V(R) is close to zero for every stored R.
for spin in (1, 2):
    far_max, far_sq, far_count = 0.0, 0.0, 0
    for R, matrix in V[spin].items():
        # Work on a copy: subtracting c_sigma here is an audit, not yet the
        # permanent alignment applied below.
        block = matrix[np.ix_(far_host_orbs, far_host_orbs)].copy()
        if R == (0, 0, 0):
            block -= cs[spin] * np.eye(len(far_host_orbs))

        abs_block = np.abs(block)
        far_max = max(far_max, float(np.max(abs_block)))
        far_sq += float(np.sum(abs_block**2))
        far_count += abs_block.size

    far_rms = float(np.sqrt(far_sq / far_count))
    print(
        f"BASIS AUDIT spin {spin}: far-host residual max/rms = "
        f"{far_max:.4e}/{far_rms:.4e} eV"
    )
    assert far_max < 0.10 and far_rms < 1e-3, (
        "far-field twin cancellation failed; common Wannier gauge is doubtful"
    )

print(
    "BASIS AUDIT PASSED "
    "(structural/centre/far-field checks; no overlap-matrix proof)"
)

# Permanently remove only the spin-independent Fe/pristine energy-zero offset.
c_common = 0.5 * (cs[1] + cs[2])
print(f"common Fe/pristine alignment constant c = {c_common:+.6f} eV")
for spin in (1, 2):
    V[spin][(0, 0, 0)] -= c_common * np.eye(181)
print("alignment applied to all 181 onsite diagonals in both spins")


far-Ta sites (d >= 5.5 A): 9 -> [(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 3), (2, 0), (3, 0), (3, 1)]
spin 1: c_sigma = +0.331143 eV
spin 2: c_sigma = +0.325119 eV
BASIS AUDIT far-host set: 26 atoms, 96 orbitals
BASIS AUDIT spin 1: far-host residual max/rms = 4.0833e-02/2.3741e-04 eV
BASIS AUDIT spin 2: far-host residual max/rms = 2.7170e-02/1.6154e-04 eV
BASIS AUDIT PASSED (structural/centre/far-field checks; no overlap-matrix proof)
common Fe/pristine alignment constant c = +0.328131 eV
alignment applied to all 181 onsite diagonals in both spins


## 5. Determine the clean-host chemical potential

The chemical potential is fixed by band counting in the nonmagnetic 11-band
host: each spin channel contains 6.5 electrons per primitive cell. On a
$120\times120$ $\mathbf{k}$ mesh, the code repeatedly evaluates

$$
N(\mu)=\frac{1}{N_k}\sum_{\mathbf{k},n}
\Theta\!\left(\mu-\varepsilon_{n\mathbf{k}}\right)
$$

and uses 80 bisection steps to solve $N(\mu)=6.5$.

$\mu$ belongs to the energy reference of `H_host` and later sets the upper limit
of the zero-temperature Lloyd/Krein integral. It is not a 4x4-supercell Fermi
level and is not the same quantity as the alignment constant `c_common`.

### Equation notation

| Symbol | Meaning |
|---|---|
| $N(\mu)$ | Mean number of occupied host bands per spin and primitive cell at chemical potential $\mu$ |
| $N_k$ | Total number of sampled k points; here $N_k=120^2$ for this solver |
| $\mathbf k$, $n$ | Fractional reciprocal-space point and host-band index |
| $\varepsilon_{n\mathbf k}$ | Energy of band $n$ at k point $\mathbf k$ |
| $\Theta(x)$ | Heaviside step function: one for an occupied level ($x>0$), otherwise zero |

### Variable-name convention

| Name | Meaning |
|---|---|
| `FILLING` | Required electrons per spin and primitive host cell (`6.5`) |
| `H_host` | Re-homed 11-orbital clean-host real-space Hamiltonian |
| `nk` | Linear k-grid size used only by the chemical-potential solver |
| `eig2` | Array of 11 host eigenvalues at every chemical-potential k point |
| `mu` | Host chemical potential satisfying the requested filling |


In [8]:
# Cell 7: host chemical potential from band counting
def Hk_of(H, k):
    """Fourier transform a real-space Hamiltonian at one fractional k point.

    Inputs:
        H (dict): Real-space blocks ``{R: H(R)}``.
        k (tuple): Two fractional reciprocal coordinates ``(kx,ky)``.

    Returns:
        numpy.ndarray: Explicitly Hermitized Bloch Hamiltonian ``H(k)``.
    """
    Hk = np.zeros_like(next(iter(H.values())))
    for R, matrix in H.items():
        phase = np.exp(2j * np.pi * (R[0] * k[0] + R[1] * k[1]))
        Hk += phase * matrix
    return 0.5 * (Hk + Hk.conj().T)


def host_mu(H1, n_per_spin=FILLING, nk=120):
    """Find the zero-temperature host chemical potential by band counting.

    Inputs:
        H1 (dict): Primitive-host real-space Hamiltonian.
        n_per_spin (float): Required electrons per spin and primitive cell.
        nk (int): Linear size of the uniform two-dimensional k mesh.

    Returns:
        float: Chemical potential in eV for which the mean number of host
        eigenvalues below ``mu`` equals ``n_per_spin``.
    """
    eigenvalues = [
        np.linalg.eigvalsh(Hk_of(H1, (i / nk, j / nk)))
        for i in range(nk)
        for j in range(nk)
    ]
    eig2 = np.asarray(eigenvalues)
    lower, upper = float(eig2.min()), float(eig2.max())

    # The occupation count is monotone in mu, so bisection is deterministic.
    for _ in range(80):
        trial_mu = 0.5 * (lower + upper)
        occupation = float(np.mean(np.sum(eig2 < trial_mu, axis=1)))
        if occupation < n_per_spin:
            lower = trial_mu
        else:
            upper = trial_mu
    return 0.5 * (lower + upper)


mu = host_mu(H_host)
print(f"host chemical potential mu = {mu:+.5f} eV")


host chemical potential mu = -4.03110 eV


## 6. Local scattering cluster and direct periodic restriction

An $L_T\times L_T$ primitive-cell grid is used only as a real-space bookkeeping
canvas. It is not an $L_T\times L_T$ Hamiltonian and is never diagonalized. The
central Fe is placed at $(L_T/2,L_T/2)$, and all complete host atoms within the
three-dimensional cutoff `RCUT` are retained. With the current parameters this
selects 7 Ta and 15 S atoms, or 80 host orbitals.

Each selected bookkeeping atom is mapped to a source atom inside the original
$4\times4$ Fe supercell and an integer $4\times4$ image quotient $\mathbf{q}$.
For a row atom $i$ and column atom $j$, matrix blocks are read with

$$
\mathbf{R}=\mathbf{q}_j-\mathbf{q}_i.
$$

The central Fe d block uses only $\mathbf{R}=\mathbf{0}$, so
direct couplings between periodic Fe images are discarded.

For the present geometry, all 22 selected atoms have $\mathbf{q}=(0,0)$ because the
cutoff sphere remains inside the source 4x4 cell. Thus the current extraction
reduces to subblocks of $V(\mathbf{0})$; the more general quotient logic is retained for
other cutoffs, impurity positions, and supercell sizes.

The same 80-orbital order is reused later when assembling the host Green
function, ensuring that potential and propagator indices refer to identical
cluster orbitals.

This real-space placement is the impurity-embedding stage: the extracted local
perturbation is expressed in a host bookkeeping grid that can be enlarged to a
different $L_T\times L_T$ environment. It is distinct from the zero-padding
operation $\mathcal P$ used only to subtract the two $4\times4$ twins.

### Equation notation

| Symbol | Meaning |
|---|---|
| $L_T$ | Linear size of the temporary primitive-cell bookkeeping grid |
| $i$, $j$ | Selected row and column atoms (or their associated orbital blocks) in that grid |
| $\mathbf q_i$, $\mathbf q_j$ | Integer quotients identifying which periodic image of the source $4\times4$ cell contains atoms $i$ and $j$ |
| $\mathbf R=\mathbf q_j-\mathbf q_i$ | Source-Hamiltonian translation required for the corresponding row/column block |
| $h$, $d$ | Host-cluster and explicit Fe-d subspaces used in block labels such as $H_{hd}$ |

### Variable-name convention

| Name | Meaning |
|---|---|
| `LT` | Linear size of the primitive-cell bookkeeping grid (`18`) |
| `RCUT` | Three-dimensional Fe-centered cluster radius in Angstrom |
| `cluster` | Ordered list of selected global bookkeeping-orbital indices |
| `local` | Map from a global bookkeeping index to a compact cluster index `0:ndim` |
| `parts` | Per-spin tuple `(dHhh, Hhd, Hdh, Hdd)` extracted in the 80+5 local basis |


In [9]:
# Cell 8: cluster construction, direct restriction, and Hermiticity check
def build_host_atoms(L):
    """Build the primitive-host bookkeeping grid in fixed orbital order.

    Inputs:
        L (int): Linear number of primitive cells in each in-plane direction.

    Returns:
        tuple: ``(atoms, index_by_cell)``. ``atoms`` contains all Ta atoms,
        followed by all bottom S and all top S atoms in row-major cell order.
        ``index_by_cell[(typ,ca,cb)]`` returns an index into ``atoms``.
    """
    host_atoms = []
    next_orbital = 0
    for typ, orbitals_per_atom in (("Ta", 5), ("S_bot", 3), ("S_top", 3)):
        for cell_index in range(L * L):
            ca, cb = divmod(cell_index, L)
            host_atoms.append(
                Atom(
                    typ,
                    ca,
                    cb,
                    next_orbital,
                    next_orbital + orbitals_per_atom,
                )
            )
            next_orbital += orbitals_per_atom

    index_by_cell = {
        (atom.typ, atom.ca, atom.cb): index
        for index, atom in enumerate(host_atoms)
    }
    return host_atoms, index_by_cell


atoms, idx_by_cell = build_host_atoms(LT)
center = np.array([LT // 2, LT // 2])
fe_center_pos = position_3d("Fe", center)

# Select complete host-atom orbital blocks within the three-dimensional cutoff.
cluster = []
for atom in atoms:
    atom_cell = np.array([atom.ca, atom.cb])
    atom_position = position_3d(atom.typ, atom_cell)
    distance_to_fe = np.linalg.norm(atom_position - fe_center_pos)
    if distance_to_fe <= RCUT + 1e-8:
        cluster.extend(range(atom.start, atom.stop))

# Convert sparse global bookkeeping indices to compact 0:ndim cluster indices.
local = {global_index: local_index for local_index, global_index in enumerate(cluster)}
ndim = len(cluster)
print(f"cluster: {ndim} host orbitals")
assert ndim == 80

# Source-atom lookup inside the original 181-orbital Fe@4x4 basis.
src_by_key = {
    (atom.typ, atom.ca, atom.cb): atom
    for atom in ATOM4
    if atom.typ != "Fe"
}
fe_src = next(atom for atom in ATOM4 if atom.typ == "Fe")


def source_of(atom):
    """Map a bookkeeping atom to its 4x4 source atom and image quotient.

    Inputs:
        atom (Atom): Host atom in the LT x LT bookkeeping grid.

    Returns:
        tuple: ``(source_atom, q)`` where ``source_atom`` is the same species at
        remainder coordinate ``r`` in the source 4x4 cell and integer ``q``
        satisfies ``raw = 4*q + r`` componentwise.
    """
    raw = np.array([atom.ca, atom.cb]) - center + np.array([2, 2])
    quotient, remainder = np.divmod(raw, 4)
    key = (atom.typ, int(remainder[0]), int(remainder[1]))
    return src_by_key[key], quotient


def direct_extract(Vd):
    """Restrict a re-homed periodic Fe potential to the local 80+5 basis.

    Inputs:
        Vd (dict): Real-space potential ``{R: V(R)}`` with complex
            ``(181,181)`` blocks in the Fe@4x4 basis.

    Returns:
        tuple: ``(dHhh, Hhd, Hdh, Hdd)`` with shapes ``(80,80)``, ``(80,5)``,
        ``(5,80)``, and ``(5,5)``. The current cutoff lies inside the source
        4x4 cell, so all selected image quotients are zero; the general quotient
        expressions are retained for portability.
    """
    selected_atoms = [
        atom
        for atom in atoms
        if atom.start < atom.stop
        and all(
            orbital in local
            for orbital in range(atom.start, atom.stop)
        )
    ]

    zero181 = np.zeros((181, 181), dtype=complex)
    dHhh = np.zeros((ndim, ndim), dtype=complex)
    Hhd = np.zeros((ndim, 5), dtype=complex)
    Hdh = np.zeros((5, ndim), dtype=complex)

    # Keep only the central Fe d block; direct periodic Fe-image couplings are
    # not part of the isolated-impurity cluster model.
    V0 = Vd.get((0, 0, 0), zero181)
    Hdd = V0[
        fe_src.start:fe_src.stop,
        fe_src.start:fe_src.stop,
    ].copy()

    for atom_i in selected_atoms:
        source_i, quotient_i = source_of(atom_i)
        rows = [
            local[orbital]
            for orbital in range(atom_i.start, atom_i.stop)
        ]

        # R always equals the column image quotient minus the row quotient.
        R_i_to_fe = (-int(quotient_i[0]), -int(quotient_i[1]), 0)
        Hhd[np.ix_(rows, range(5))] = Vd.get(R_i_to_fe, zero181)[
            source_i.start:source_i.stop,
            fe_src.start:fe_src.stop,
        ]

        R_fe_to_i = (int(quotient_i[0]), int(quotient_i[1]), 0)
        Hdh[np.ix_(range(5), rows)] = Vd.get(R_fe_to_i, zero181)[
            fe_src.start:fe_src.stop,
            source_i.start:source_i.stop,
        ]

        for atom_j in selected_atoms:
            source_j, quotient_j = source_of(atom_j)
            cols = [
                local[orbital]
                for orbital in range(atom_j.start, atom_j.stop)
            ]
            R_i_to_j = (
                int(quotient_j[0] - quotient_i[0]),
                int(quotient_j[1] - quotient_i[1]),
                0,
            )
            dHhh[np.ix_(rows, cols)] = Vd.get(R_i_to_j, zero181)[
                source_i.start:source_i.stop,
                source_j.start:source_j.stop,
            ]

    return dHhh, Hhd, Hdh, Hdd


parts = {}
for spin in (1, 2):
    dHhh, Hhd, Hdh, Hdd = direct_extract(V[spin])
    anti = float(np.max(np.abs(dHhh - dHhh.conj().T)))
    antid = float(np.max(np.abs(Hdh - Hhd.conj().T)))
    print(
        f"cluster Hermiticity check (spin {spin}): residuals "
        f"dHhh {anti:.2e}, Hhd/Hdh {antid:.2e} eV (no averaging applied)"
    )
    assert anti < 1e-12 and antid < 1e-12
    parts[spin] = (dHhh, Hhd, Hdh, Hdd)
print("cluster Hermiticity check passed")


cluster: 80 host orbitals
cluster Hermiticity check (spin 1): residuals dHhh 0.00e+00, Hhd/Hdh 0.00e+00 eV (no averaging applied)
cluster Hermiticity check (spin 2): residuals dHhh 0.00e+00, Hhd/Hdh 0.00e+00 eV (no averaging applied)
cluster Hermiticity check passed


## 7. Dense-k spectral representation of the clean-host Green function

The clean 11-orbital Hamiltonian is diagonalized on an $N_K\times N_K$ mesh:

$$
H_{\mathrm{host}}(\mathbf{k})=U(\mathbf{k})
\operatorname{diag}\!\left[\varepsilon_n(\mathbf{k})\right]
U^{\dagger}(\mathbf{k}).
$$

For $z=E+i\eta$, the reciprocal-space resolvent is

$$
G(\mathbf{k},z)=U(\mathbf{k})
\operatorname{diag}\!\left[\frac{1}{z-\varepsilon_n(\mathbf{k})}\right]
U^{\dagger}(\mathbf{k}).
$$

The real-space propagator between primitive cells separated by $\boldsymbol{\Delta}$ is

$$
F(\boldsymbol{\Delta},z)=\frac{1}{N_k}
\sum_{\mathbf{k}}e^{+2\pi i\,\mathbf{k}\cdot\boldsymbol{\Delta}}
G(\mathbf{k},z).
$$

The 18x18 bookkeeping grid contributes only the primitive-cell coordinates of
the 80 selected cluster orbitals. The code tabulates the unique differences
needed for propagation within one cluster and between two clusters separated
by each value in `OFFSETS`. Duplicate differences share the same Fourier sum.

### Equation notation

| Symbol | Meaning |
|---|---|
| $U(\mathbf k)$, $U^\dagger(\mathbf k)$ | Matrix of host eigenvectors and its Hermitian conjugate |
| $\varepsilon_n(\mathbf k)$, $n$ | Host eigenvalue and band index |
| $z=E+i\eta$ | Complex retarded energy; $E$ is real and $\eta>0$ is the numerical broadening `ETA` |
| $G(\mathbf k,z)$ | Clean-host reciprocal-space resolvent $(zI-H_{\mathrm{host}})^{-1}$ |
| $F(\boldsymbol\Delta,z)$, $\boldsymbol\Delta$, $N_k$ | Real-space propagator, primitive-cell separation, and total number of dense-mesh k points |

### Variable-name convention

| Name | Meaning |
|---|---|
| `Rs`, `HR` | Parallel arrays of host lattice vectors and 11x11 real-space blocks |
| `kf` | Flattened $N_K^2\times2$ array of fractional reciprocal coordinates |
| `eps`, `Uk` | Host band energies and eigenvectors on the dense k mesh |
| `uniq`, `idx` | Unique required cell differences and lookup tables for cluster pairs |
| `P` | Precomputed phase matrix $e^{2\pi i\,\mathbf{k}\cdot\boldsymbol{\Delta}}$ for every `k` and unique `Delta` |


In [10]:
# Cell 9: dense-k eigendecomposition and host propagator
Rs = np.array(list(H_host.keys()), dtype=int)
HR = np.array([H_host[tuple(R)] for R in Rs])

# Uniform fractional reciprocal mesh, flattened to one k point per row.
ii, jj = np.meshgrid(
    np.arange(NK) / NK,
    np.arange(NK) / NK,
    indexing="ij",
)
kf = np.column_stack([ii.ravel(), jj.ravel()])

# Diagonalize H(k) in batches to control peak memory use.
eps = np.empty((NK * NK, 11))
Uk = np.empty((NK * NK, 11, 11), dtype=complex)
for i0 in range(0, NK * NK, 4096):
    k_batch = kf[i0:i0 + 4096]
    phases = np.exp(2j * np.pi * (k_batch @ Rs[:, :2].T))
    Hk = np.einsum("mr,rij->mij", phases, HR, optimize=True)
    Hk = 0.5 * (Hk + np.conj(np.swapaxes(Hk, 1, 2)))
    eigenvalues, eigenvectors = np.linalg.eigh(Hk)
    eps[i0:i0 + 4096] = eigenvalues
    Uk[i0:i0 + 4096] = eigenvectors
print(f"dense mesh: {NK}x{NK} k-points diagonalized")

# For each compact cluster orbital, record its primitive-cell coordinate and
# its orbital index alpha in the 11-orbital primitive-host basis.
cells = np.array([
    [atom.ca, atom.cb]
    for atom in atoms
    for orbital in range(atom.start, atom.stop)
    if orbital in local
])
alphas = np.array([
    TYPE_WF_1[atom.typ][0] + (orbital - atom.start)
    for atom in atoms
    for orbital in range(atom.start, atom.stop)
    if orbital in local
])

# Delta_ij = cell_i - cell_j for one cluster. Additional +/-n shifts provide
# propagation between two translated copies of that cluster.
d0 = (cells[:, None, :] - cells[None, :, :]).reshape(-1, 2)
shifts = [(0, 0)] + [
    shift
    for n in OFFSETS
    for shift in ((-n, 0), (n, 0))
]
all_d = np.concatenate([d0 + np.asarray(shift) for shift in shifts], axis=0)

# Evaluate each distinct cell difference only once, and retain an 80x80 lookup
# table for every requested cluster shift.
uniq, inverse = np.unique(all_d, axis=0, return_inverse=True)
idx = {
    shift: inverse[
        block * ndim * ndim:(block + 1) * ndim * ndim
    ].reshape(ndim, ndim)
    for block, shift in enumerate(shifts)
}

# Fourier phases depend on k and Delta but not on the energy z.
P = np.exp(2j * np.pi * (kf @ uniq.T.astype(float)))
print(f"{len(uniq)} unique cell differences tabulated")


def greens_F(z):
    """Evaluate the clean-host real-space propagator for all required Delta.

    Inputs:
        z (complex): Retarded complex energy, normally ``E + 1j*ETA``.

    Returns:
        numpy.ndarray: Array with shape ``(nDelta,11,11)`` containing
        ``F(Delta,z)`` in the primitive-host orbital basis.
    """
    band_resolvent = 1.0 / (z - eps)
    Gk = np.einsum(
        "kab,kb,kcb->kac",
        Uk,
        band_resolvent,
        np.conj(Uk),
        optimize=True,
    )
    flattened = Gk.reshape(Gk.shape[0], -1)
    return (P.T @ flattened).reshape(P.shape[1], 11, 11) / eps.shape[0]


print("host propagator ready")


dense mesh: 180x180 k-points diagonalized
121 unique cell differences tabulated
host propagator ready


## 8. Two-impurity scattering, Lloyd phases, and exchange $J(n)$

At each energy, the five explicit Fe d orbitals are Schur-downfolded into the
80-orbital host cluster:

$$
V_{\mathrm{eff},\sigma}(E)=\Delta H_{hh,\sigma}
+H_{hd,\sigma}\left(zI-H_{dd,\sigma}\right)^{-1}H_{dh,\sigma}.
$$

The single-impurity $T$ matrix

$$
T_{\sigma}=\left(I-V_{\mathrm{eff},\sigma}G_{AA}^{0}\right)^{-1}
V_{\mathrm{eff},\sigma}
$$

resums repeated scattering at cluster A. For a second cluster B separated by
$n\mathbf{a}_1$, its local propagator dressed by A is

$$
G_{BB,\sigma}=G_{BB}^{0}+G_{BA}^{0}T_{\sigma}G_{AB}^{0}.
$$

For magnetic alignment $c$ and separation $n\mathbf a_1$, the pair contribution
to the zero-temperature grand potential is

$$
\Omega_c(n)=\frac{1}{\pi}\sum_{\sigma\in\{\uparrow,\downarrow\}}
\int_{E_{\min}}^{\mu}\!\mathrm dE\;\operatorname{Im}
\ln\det\!\left[I-T_A^\sigma G_{AB}^0
T_{B,c}^\sigma G_{BA}^0\right],
\qquad c\in\{\mathrm{FM},\mathrm{AP}\}.
$$

FM uses $T_{B,\mathrm{FM}}^\sigma=T^\sigma$. AP reverses the moment on B, so
$T_{B,\mathrm{AP}}^\uparrow=T^\downarrow$ and
$T_{B,\mathrm{AP}}^\downarrow=T^\uparrow$. The code evaluates the equivalent
sequential-scattering determinant

$$
\ln\det\!\left[I-V_{B,c}^\sigma G_{BB}^{A,\sigma}\right]
=\ln\det\!\left[I-V_{B,c}^\sigma G_{BB}^0\right]
+\ln\det\!\left[I-T_A^\sigma G_{AB}^0
T_{B,c}^\sigma G_{BA}^0\right].
$$

The first term is the isolated-B contribution. Its spin sum is identical in FM
and AP and therefore cancels in $\Omega_{\mathrm{AP}}-\Omega_{\mathrm{FM}}$.
The imaginary log-determinant phases are unwrapped along the energy axis before
integration. With the Heisenberg convention

$$
\mathcal{H}=-J\,\mathbf{e}_1\cdot\mathbf{e}_2,\qquad
J=\frac{\Omega_{\mathrm{AP}}-\Omega_{\mathrm{FM}}}{2},
$$

the numerical factor $500$ converts $\mathrm{eV}$ to $\mathrm{meV}$ and applies
the factor one half.

### Equation notation

| Symbol | Meaning |
|---|---|
| $A$, $B$, $n\mathbf a_1$ | First impurity, second impurity, and their real-space separation |
| $\sigma$, $c$ | Electron spin channel and moment alignment (`FM` or antiparallel `AP`) |
| $T_A^\sigma$, $T_{B,c}^\sigma$ | Single-impurity T matrices at A and B for the indicated spin potential |
| $G_{XY}^0$ | Clean-host propagator from cluster $Y$ to cluster $X$; the first subscript is the row/destination cluster |
| $E_{\min}$, $\mu$, $\Omega_c(n)$ | Lower integration cutoff `mu-7.5`, upper cutoff/chemical potential, and pair grand-potential contribution |

### Variable-name convention

| Name | Meaning |
|---|---|
| `energies` | Uniform integration grid from `mu-7.5 eV` through `mu` |
| `parts` | Per-spin local blocks `(dHhh,Hhd,Hdh,Hdd)` |
| `GAA`, `GAB`, `GBA` | Intra-cluster and directed inter-cluster clean-host propagators |
| `Vup`, `Vdn` | Spin-dependent downfolded 80x80 effective potentials |
| `Tup`, `Tdn` | Single-impurity T matrices that resum repeated scattering |


In [11]:
# Cell 10: energy integration and exchange J(n)
energies = np.linspace(mu - 7.5, mu, NE)
eye = np.eye(ndim, dtype=complex)


def veff_of(E, spin):
    """Downfold the five Fe d orbitals into an 80x80 host-space potential.

    Inputs:
        E (float): Real energy in eV, in the same reference as ``H_host``.
        spin (int): Spin-channel key, either 1 or 2.

    Returns:
        numpy.ndarray: Energy-dependent complex ``(80,80)`` effective
        potential ``dHhh + Hhd (z-Hdd)^-1 Hdh``.
    """
    dHhh, Hhd, Hdh, Hdd = parts[spin]
    z = complex(float(E), ETA)
    gd = np.linalg.inv(z * np.eye(5, dtype=complex) - Hdd)
    return dHhh + Hhd @ gd @ Hdh


def complex_logdet(matrix):
    """Compute a numerically stable principal complex log determinant.

    Inputs:
        matrix (numpy.ndarray): Square complex matrix.

    Returns:
        complex: ``log(det(matrix))`` on the principal branch. The energy-axis
        phase is unwrapped after all samples have been collected.
    """
    phase, log_abs_det = np.linalg.slogdet(matrix)
    return np.log(phase) + log_abs_det


# Store the sequential-scattering log determinants. Each entry contains a
# common isolated-B term plus the A-B pair term; the common term cancels in
# the spin-summed AP-FM difference documented above.
logs = {
    (n, alignment): []
    for n in OFFSETS
    for alignment in ("fm", "ap")
}

for E in energies:
    # Clean-host propagator inside one cluster.
    F = greens_F(complex(float(E), ETA))
    GAA = F[idx[(0, 0)], alphas[:, None], alphas[None, :]]

    Vup, Vdn = veff_of(E, 1), veff_of(E, 2)

    # T = (I - V G0)^-1 V sums repeated scattering from one impurity.
    Tup = np.linalg.solve(eye - Vup @ GAA, Vup)
    Tdn = np.linalg.solve(eye - Vdn @ GAA, Vdn)

    for n in OFFSETS:
        # Propagation between cluster A and a translated cluster B.
        GAB = F[idx[(-n, 0)], alphas[:, None], alphas[None, :]]
        GBA = F[idx[(n, 0)], alphas[:, None], alphas[None, :]]

        # Local propagator at B after scattering from impurity A.
        Gup = GAA + GBA @ Tup @ GAB
        Gdn = GAA + GBA @ Tdn @ GAB

        # FM uses the same spin potential at A and B; AP swaps the potential
        # assigned to impurity B while keeping both electron-spin channels.
        logs[(n, "fm")].append(
            complex_logdet(eye - Vup @ Gup)
            + complex_logdet(eye - Vdn @ Gdn)
        )
        logs[(n, "ap")].append(
            complex_logdet(eye - Vdn @ Gup)
            + complex_logdet(eye - Vup @ Gdn)
        )

# The imaginary log determinant is a scattering phase. Remove artificial
# 2*pi branch jumps before integrating the AP-FM grand-potential difference.
J = {}
for n in OFFSETS:
    phase_fm = np.unwrap(np.imag(np.asarray(logs[(n, "fm")])))
    phase_ap = np.unwrap(np.imag(np.asarray(logs[(n, "ap")])))
    J[n] = (
        500.0
        * (trapezoid(phase_ap, energies) - trapezoid(phase_fm, energies))
        / np.pi
    )
    # 500 = 1000 meV/eV times 1/2 from J=(Omega_AP-Omega_FM)/2.
    print(f"J(n={n}) = {J[n]:+9.5f} meV")


J(n=2) =  +2.56448 meV
J(n=3) =  -0.32799 meV
J(n=4) =  +0.16747 meV
J(n=5) =  -0.20100 meV
J(n=6) =  +0.09560 meV
